# Final PCR Prediction (Classification)

In [437]:
import pandas as pd
import numpy as np

import joblib, csv

### Preprocessing

#### Before Preprocessing

In [438]:
dataset_x_all = pd.read_csv("../datasets/FinalTestDataset2025.csv")
dataset_x_all

,ID,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002219,47.0,1,1,0,0,3,2,1,1,...,0.498350,0.498350,3.144594,0.003447,8.257693e+06,150.048587,0.001753,0.037110,0.001369,0.001513
1,TRG002222,41.0,1,1,0,0,3,2,1,0,...,0.622381,0.622381,2.061654,0.006535,1.568442e+06,26.484938,0.009649,0.019352,0.000321,0.008285
2,TRG002223,53.0,0,0,0,1,2,1,1,1,...,0.412482,0.412482,3.440353,0.005391,2.656925e+06,174.606929,0.001594,0.075152,0.005255,0.001444
3,TRG002235,46.0,1,1,0,0,2,1,1,1,...,0.378333,0.378333,3.531715,0.007102,1.714787e+06,96.787378,0.002772,0.053377,0.002666,0.002406
4,TRG002240,39.0,0,0,1,0,2,2,1,1,...,0.524767,0.524767,2.186214,0.007896,5.104793e+05,12.789071,0.020072,0.023140,0.000463,0.017172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,TRG002953,62.3,0,0,1,0,2,1,1,0,...,0.550099,0.550097,2.632664,0.004373,1.583970e+06,91.084710,0.002972,0.042656,0.002168,0.002828
129,TRG002955,46.1,0,0,0,1,3,3,1,1,...,0.439568,0.439568,3.056046,0.001339,1.671271e+07,79.989003,0.003282,0.024716,0.000812,0.003078
130,TRG002956,44.3,0,0,0,1,3,2,1,0,...,0.450758,0.450758,2.820884,0.003774,4.493563e+06,144.506524,0.001912,0.053334,0.003453,0.001759
131,TRG002957,37.1,0,0,0,1,3,3,1,1,...,0.384860,0.384860,3.680654,0.003699,4.054944e+06,250.108074,0.001168,0.091763,0.008428,0.001107


#### Type-specific preprocessing

In [439]:
ids = dataset_x_all["ID"]
dataset_x_all = dataset_x_all.drop("ID",axis=1)
dataset_x_all['original_shape_VoxelVolume'] = dataset_x_all['original_shape_VoxelVolume'].astype(np.float64)
categorical_names = dataset_x_all.select_dtypes(np.int64).columns

continuous_names = dataset_x_all.select_dtypes(np.float64).columns

dataset_x_all = dataset_x_all.replace([999], np.nan)

#### General Preprocessing

In [440]:
Xs = dataset_x_all.copy()

lower_quartile = Xs[continuous_names].quantile(0.25)
upper_quartile = Xs[continuous_names].quantile(0.75)
iqr = upper_quartile - lower_quartile

lower_bound = lower_quartile - 1.5 * iqr
upper_bound = upper_quartile + 1.5 * iqr

Xs[continuous_names] = np.where(Xs[continuous_names] < lower_bound, lower_bound, Xs[continuous_names])
Xs[continuous_names] = np.where(Xs[continuous_names] > upper_bound, upper_bound, Xs[continuous_names])

In [441]:
categorical_ranges = {
    "pCR (outcome)": (0,1),
    "ER": (0, 1),
    "PgR": (0, 1),
    "HER2": (0, 1),
    "TrippleNegative": (0, 1),
    "ChemoGrade": (1, 3),
    "Proliferation": (1, 3),
    "HistologyType": (1, 2),
    "LNStatus": (0, 1),
    "TumourStage": (1, 4),
    "Gene": (0, 1)
}

In [442]:
data_train_cat = pd.concat([Xs[categorical_names]], axis=1)
for col in data_train_cat.columns:
    
    # Truncate data to be within range for categorical features
    data_train_cat[col] = np.where(data_train_cat[col] > categorical_ranges[col][1], categorical_ranges[col][1],  data_train_cat[col])
    data_train_cat[col] = np.where(data_train_cat[col] < categorical_ranges[col][0], categorical_ranges[col][0],  data_train_cat[col])

In [443]:
imputer_knn = joblib.load('../exports/pcr_imputer_knn.pkl')
z_scaler = joblib.load('../exports/pcr_z_scaler_knn.pkl')

Xs = pd.DataFrame(imputer_knn.transform(Xs), columns=Xs.columns, index=Xs.index)
Xs[continuous_names] = z_scaler.transform(Xs[continuous_names])
Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,-0.424396,1.0,1.0,0.0,0.0,3.0,2.0,1.0,1.0,2.0,...,0.697776,0.741109,0.618414,0.050050,0.244878,0.340129,-0.781116,-0.532427,-0.738090,-0.778895
1,-0.999802,1.0,1.0,0.0,0.0,3.0,2.0,1.0,0.0,2.0,...,1.543117,1.613684,-0.974286,1.357938,-0.641530,-0.763377,0.090974,-0.941931,-0.915005,0.240547
2,0.151011,0.0,0.0,0.0,1.0,2.0,1.0,1.0,1.0,2.0,...,0.112536,0.137015,1.053393,0.873464,-0.497292,0.559452,-0.798623,0.344820,-0.082155,-0.789384
3,-0.520297,1.0,1.0,0.0,0.0,2.0,1.0,1.0,1.0,3.0,...,-0.120208,-0.103228,1.187762,1.597793,-0.622137,-0.135529,-0.668562,-0.157327,-0.519148,-0.644522
4,-1.191604,0.0,0.0,1.0,0.0,2.0,2.0,1.0,1.0,2.0,...,0.877824,0.926958,-0.791093,1.934221,-0.781723,-0.885690,1.242226,-0.854590,-0.891120,1.578439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,1.042890,0.0,0.0,1.0,0.0,2.0,1.0,1.0,0.0,3.0,...,1.050475,1.105159,-0.134490,0.442284,-0.639472,-0.186458,-0.646468,-0.404553,-0.603225,-0.581045
129,-0.510706,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,4.0,...,0.297143,0.327569,0.488186,-0.843143,1.365271,-0.285550,-0.612194,-0.818239,-0.832115,-0.543320
130,-0.683328,0.0,0.0,0.0,1.0,3.0,2.0,1.0,0.0,2.0,...,0.373407,0.406290,0.142329,0.188376,-0.253915,0.290635,-0.763476,-0.158312,-0.386357,-0.741882
131,-1.373816,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,3.0,...,-0.075727,-0.057314,1.406809,0.156429,-0.312038,1.233728,-0.845669,0.727865,0.453610,-0.840126


In [444]:
# Convert categoricals back to int64 (no more NaN/float64) --------------------------------------------------
Xs[categorical_names] = Xs[categorical_names].astype(np.int64)
Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,-0.424396,1,1,0,0,3,2,1,1,2,...,0.697776,0.741109,0.618414,0.050050,0.244878,0.340129,-0.781116,-0.532427,-0.738090,-0.778895
1,-0.999802,1,1,0,0,3,2,1,0,2,...,1.543117,1.613684,-0.974286,1.357938,-0.641530,-0.763377,0.090974,-0.941931,-0.915005,0.240547
2,0.151011,0,0,0,1,2,1,1,1,2,...,0.112536,0.137015,1.053393,0.873464,-0.497292,0.559452,-0.798623,0.344820,-0.082155,-0.789384
3,-0.520297,1,1,0,0,2,1,1,1,3,...,-0.120208,-0.103228,1.187762,1.597793,-0.622137,-0.135529,-0.668562,-0.157327,-0.519148,-0.644522
4,-1.191604,0,0,1,0,2,2,1,1,2,...,0.877824,0.926958,-0.791093,1.934221,-0.781723,-0.885690,1.242226,-0.854590,-0.891120,1.578439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,1.042890,0,0,1,0,2,1,1,0,3,...,1.050475,1.105159,-0.134490,0.442284,-0.639472,-0.186458,-0.646468,-0.404553,-0.603225,-0.581045
129,-0.510706,0,0,0,1,3,3,1,1,4,...,0.297143,0.327569,0.488186,-0.843143,1.365271,-0.285550,-0.612194,-0.818239,-0.832115,-0.543320
130,-0.683328,0,0,0,1,3,2,1,0,2,...,0.373407,0.406290,0.142329,0.188376,-0.253915,0.290635,-0.763476,-0.158312,-0.386357,-0.741882
131,-1.373816,0,0,0,1,3,3,1,1,3,...,-0.075727,-0.057314,1.406809,0.156429,-0.312038,1.233728,-0.845669,0.727865,0.453610,-0.840126


Notes: HistologyType 1.5????????????????????????????????????????????????????????????????????????? LMAO

### Feature Selection

In [445]:
feature_sel = joblib.load('../exports/selected_features.pkl')
feature_sel

['ER',
 'PgR',
 'TrippleNegative',
 'LNStatus',
 'Gene',
 'original_firstorder_10Percentile',
 'original_firstorder_InterquartileRange',
 'original_firstorder_Variance',
 'original_glcm_DifferenceVariance',
 'original_glcm_Imc1',
 'HER2']

In [446]:
Xs = Xs[feature_sel]
Xs

,ER,PgR,TrippleNegative,LNStatus,Gene,original_firstorder_10Percentile,original_firstorder_InterquartileRange,original_firstorder_Variance,original_glcm_DifferenceVariance,original_glcm_Imc1,HER2
0,1,1,0,1,1,0.165766,-0.971715,-1.009008,-0.559833,0.493987,0
1,1,1,0,0,1,0.649187,-0.884291,-1.014283,-0.980754,1.176446,0
2,0,0,1,1,1,-0.290922,-0.096916,-0.086573,0.358623,0.053962,0
3,1,1,0,1,0,0.241980,-0.056130,-0.129885,-0.166369,0.780381,0
4,0,0,0,1,0,0.687473,0.115335,0.076071,-0.878595,1.227032,1
...,...,...,...,...,...,...,...,...,...,...,...
128,0,0,0,0,1,0.409198,1.219845,2.084746,-0.259279,-2.161624,1
129,0,0,1,1,1,0.838120,0.032988,0.161574,-0.719661,-2.332518,0
130,0,0,1,0,0,-0.244697,-1.333290,-1.204102,-0.081236,-1.885165,0
131,0,0,1,1,1,-0.610931,-0.238992,-0.250719,0.853813,-0.800276,0


### Prediction

In [ ]:
models: dict
models = joblib.load('../exports/pcr_models.pkl') # Load all models

# for key in models.keys():
#     print(f"{key}: {models[key].predict(Xs)}")

# Then choose a model ig
# LogReg seems to have greaatest acc in testing
    # use it as output

m = models['Logistic Regression']
m_results = m.predict(Xs)

# print(m_results)
# print(ids)


# write to csv using original ids (data order remains the same as no train/test split for example)
with open("../datasets/FinalTestDataset2025Predictions.csv", "w", encoding="utf-8", newline='') as file:
    write = pd.concat([ids, pd.DataFrame(m_results, columns=['pCR (outcome)'])], ignore_index=False, sort=False, axis=1)
    write.to_csv(file, index=False)
    
write

,ID,pCR (outcome)
0,TRG002219,0
1,TRG002222,0
2,TRG002223,0
3,TRG002235,0
4,TRG002240,1
...,...,...
128,TRG002953,1
129,TRG002955,0
130,TRG002956,0
131,TRG002957,0
